# Estimasi SOC Baterai Li-Ion Berbasis Contextual Hard-Coulomb LSTM/TCN
# dengan Pengujian Suhu Ekstrem

---

**Rangkuman Penelitian Lengkap — Dari Preprocessing hingga Deployment**

Notebook ini mendokumentasikan seluruh pipeline penelitian estimasi State of Charge (SOC) baterai Lithium-Ion menggunakan arsitektur deep learning Physics-Informed. Notebook ini bersifat **dokumentasi saja** dan tidak perlu dijalankan.

---

### Daftar Isi

1. [Pendahuluan & Latar Belakang](#1)
2. [Konfigurasi & Konstanta Fisika](#2)
3. [Evolusi Preprocessing (v1 → v5)](#3)
4. [Evolusi Arsitektur Model (v1 → v7)](#4)
5. [Ekstraksi R_int dari HPPC](#5)
6. [Pipeline Training Final](#6)
7. [Evaluasi & Metrik](#7)
8. [Penemuan Kritis: Anchor Trap −20°C](#8)
9. [Hasil Akhir & Rekomendasi Deployment](#9)
10. [Kesimpulan](#10)

---
<a id='1'></a>
## 1. Pendahuluan & Latar Belakang

### 1.1 Motivasi Penelitian

Estimasi State of Charge (SOC) yang akurat merupakan komponen kritis dalam Battery Management System (BMS) untuk kendaraan listrik. Tantangan utama meliputi:

- **Non-linearitas elektrokimia** — hubungan antara tegangan terminal dan SOC sangat non-linear, terutama pada suhu ekstrem
- **Ohmic Drop Illusion** — pada suhu rendah, resistansi internal meningkat drastis sehingga tegangan terminal tidak lagi merepresentasikan SOC sebenarnya
- **Physics Violation** — model machine learning murni seringkali menghasilkan prediksi yang melanggar hukum fisika (SOC naik saat baterai sedang discharge)

### 1.2 Dataset: Kollmeyer LG HG2

Dataset yang digunakan adalah LG 18650HG2 Li-ion Battery Data dari University of Wisconsin-Madison:

| Parameter | Nilai |
|-----------|-------|
| Cell Type | LG 18650HG2 (NMC) |
| Nominal Capacity | 3.0 Ah |
| Voltage Range | 2.5 V – 4.2 V |
| Temperatures | −20°C, −10°C, 0°C, 10°C, 25°C, 40°C |
| Drive Cycles | UDDS, LA92, HWFET, US06, Mixed |
| HPPC Profiles | 1 per temperature (R_int extraction) |
| Sampling Rate | Variable → resampled ke 1 Hz |

### 1.3 Skenario Evaluasi

| Skenario | Train | Val | Test | Tujuan |
|----------|-------|-----|------|--------|
| **A (Zero-Shot OOD)** | 25°C, 10°C | 0°C + holdout 10% | 40°C, −10°C, **−20°C** | Generalisasi suhu ekstrem |
| **B (In-Distribution)** | 70% per suhu | 10% per suhu | 20% per suhu | Performa in-distribution |

---
<a id='2'></a>
## 2. Konfigurasi & Konstanta Fisika

Seluruh hyperparameter dan konstanta fisika didefinisikan dalam `src/config.py` untuk menjaga konsistensi dan reprodusibilitas.

In [ ]:
# ============================================================
# config.py — Project-wide hyperparameters (LOCKED)
# ============================================================

# --- Konstanta Fisika Baterai ---
Q_NOMINAL   = 3.0          # Ah — kapasitas nominal LG HG2
V_MAX       = 4.2          # V  — batas atas tegangan
V_MIN       = 2.5          # V  — batas bawah tegangan

# --- Arsitektur Model ---
NUM_INPUTS      = 5                  # [V_proxy, I, T, dV/dt, dI/dt]
NUM_FILTERS     = 64                 # channels per TemporalBlock
KERNEL_SIZE     = 7                  # conv kernel size
DROPOUT         = 0.2                # dropout probability
DILATION_RATES  = [1, 2, 4, 8]       # satu per TemporalBlock

# --- Training ---
BATCH_SIZE    = 1024
LEARNING_RATE = 1e-3
EPOCHS        = 100
RANDOM_SEED   = 42

# --- Resistansi Internal per Suhu (dari HPPC) ---
# Diekstrak dari transisi REST→DCH pada file HPPC.
# R_int = ΔV / ΔI menggunakan sample pertama discharge pulse.
R_INT_PER_TEMP = {
    '40degC':  0.01651,    # 16.51 mΩ
    '25degC':  0.01986,    # 19.86 mΩ
    '10degC':  0.02875,    # 28.75 mΩ
    '0degC':   0.04008,    # 40.08 mΩ
    'n10degC': 0.06219,    # 62.19 mΩ
    'n20degC': 0.10983,    # 109.83 mΩ  ← 6.6× lebih besar dari 40°C!
}

# Threshold arus untuk Hard Constraint 3-arah
CURRENT_THRESHOLD = 0.05  # Ampere

# Physics-Informed Scaling bounds (V_proxy)
PHYS_MIN_V3 = [2.5,  -20.0, -20.0, -2.0, -20.0]
PHYS_MAX_V3 = [4.25,  20.0,  50.0,  2.0,  20.0]

### Catatan Penting tentang R_int

Perhatikan bahwa R_int pada −20°C (109.83 mΩ) adalah **6.6× lebih besar** dari R_int pada 40°C (16.51 mΩ). Ini berarti:

- Pada −20°C, arus 5A menyebabkan **voltage drop ≈ 0.55V** — hampir 15% dari total range tegangan
- Tanpa koreksi V_proxy, model "melihat" tegangan yang sudah terdistorsi oleh Ohmic drop
- Ini adalah akar dari **Ohmic Drop Illusion** yang menjadi motivasi penggunaan V_proxy

---
<a id='3'></a>
## 3. Evolusi Preprocessing (v1 → v5)

Pipeline preprocessing mengalami 5 iterasi, masing-masing mengatasi masalah fundamental yang ditemukan pada versi sebelumnya.

```
v1 (preprocessing.py)          → OCV-anchor + Capacity-based SOC
  │
  ├── Masalah: Raw voltage terdistorsi oleh Ohmic drop
  ▼
v3 (preprocessing_v3.py)       → V_proxy = V_t − I·R_int(T)
  │
  ├── Masalah: Temporal leakage antar split (sliding window overlap)
  ▼
v4 (preprocessing_v4.py)       → Strict 1Hz + Split-Before-Windowing
  │
  ├── Masalah: Anchor head lemah pada −20°C (observability bottleneck)
  ▼
v5 (preprocessing_v5_contextual.py) → Contextual 60s causal history features
```

### 3.1 Preprocessing v1: OCV-Anchor + Capacity SOC

**File**: `src/preprocessing.py`

Pipeline awal dengan fitur sederhana:
- Features: `[Voltage, Current, Temperature, dV/dt, dI/dt]`
- SOC ground truth dari kolom Capacity hardware
- SOC_initial dikalibrasi via OCV-SOC lookup (PCHIP interpolation dari HPPC rest segments)
- Sliding window: 100 timesteps, stride 10

In [ ]:
# ============================================================
# Preprocessing v1: Feature Engineering
# ============================================================
# File: src/preprocessing.py

# OCV-SOC lookup dari HPPC rest segments (≥600s)
def build_ocv_soc_lookup(temp_name):
    """
    Build PCHIP interpolator: voltage → SOC dari HPPC rest segments.
    Menggunakan rest periods ≥600s di mana tegangan sudah stabil
    untuk mengaproksimasi Open Circuit Voltage (OCV).
    """
    # 1. Load HPPC CSV
    # 2. Identifikasi rest segments (|I| < 0.01A, durasi ≥ 600s)
    # 3. Ekstrak OCV = V_rest_akhir, SOC = 1 - cap_used/Q_actual
    # 4. Build PchipInterpolator(OCV → SOC)
    pass

# Feature engineering
def engineer_features(df, q_actual, ocv_lookup):
    """
    SOC(t) = SOC_initial - |Capacity(t) - Capacity(0)| / Q_actual
    SOC_initial = OCV_lookup(V(0))  # dari HPPC calibration
    """
    # Kinetic derivatives:
    # dV/dt = diff(Voltage) / dt
    # dI/dt = diff(Current) / dt
    pass

FEATURE_COLS = ['Voltage', 'Current', 'Temperature', 'dV_dt', 'dI_dt']

# Sliding window: Seq2Seq
def build_sequences(df, window=100, stride=10):
    """
    X: (N, 100, 5) — input features
    y: (N, 100)    — full SOC trajectory per window (Seq2Seq)
    """
    pass

### 3.2 Preprocessing v3: V_proxy (Koreksi Ohmic Drop)

**File**: `src/preprocessing_v3.py`

**Inovasi kunci**: Mengganti raw voltage dengan **V_proxy** untuk menghilangkan distorsi Ohmic drop.

$$V_{proxy}(t) = V_t(t) - I(t) \times R_{int}(T)$$

Dimana $R_{int}(T)$ adalah resistansi internal yang bergantung suhu, diekstrak dari data HPPC.

**Perubahan dari v1:**
- Feature index 0: `Voltage` → `V_proxy`
- Feature index 3: `dV/dt` → `dV_proxy/dt`
- Menambahkan `I_unscaled_{split}.npy` untuk Hard Constraint layer
- Physics scaling bounds diperbarui (V_proxy range lebih sempit)

In [ ]:
# ============================================================
# Preprocessing v3: V_proxy = V_t - I * R_int(T)
# ============================================================
# File: src/preprocessing_v3.py

def engineer_features_v3(df, q_actual, r_int, ocv_lookup=None):
    """
    V_proxy menghilangkan distorsi Ohmic pada tegangan terminal.
    
    Saat discharge (I < 0):
      V_t turun karena beban + Ohmic drop
      V_proxy = V_t - I*R_int → lebih tinggi dari V_t
      V_proxy mendekati OCV sebenarnya
    
    Efeknya: model "melihat" tegangan yang lebih representatif
    terhadap SOC sebenarnya, terutama pada suhu rendah.
    """
    df['V_proxy'] = df['Voltage'] - df['Current'] * r_int
    df['dV_proxy_dt'] = (df['V_proxy'].diff() / safe_dt).clip(-2.0, 2.0)

FEATURE_COLS_V3 = ['V_proxy', 'Current', 'Temperature', 'dV_proxy_dt', 'dI_dt']

# Juga menyimpan I_unscaled untuk Hard Constraint:
# np.save("I_unscaled_train.npy", I_raw_train)
# np.save("I_unscaled_val.npy",   I_raw_val)
# np.save("I_unscaled_test.npy",  I_raw_test)

### 3.3 Preprocessing v4: Leakage-Safe 1 Hz Pipeline

**File**: `src/preprocessing_v4.py`

**Masalah yang diatasi**: Temporal leakage antara train/val/test splits.

Pada v1–v3, sliding window dibuat **sebelum** splitting. Dengan stride=10 dan window=100, window yang berdekatan saling overlap 90 timestep. Jika split dilakukan setelah windowing, data train dan val/test berbagi timestep yang sama → **leakage**.

**Solusi v4**:
1. Raw profile → **strict 1.0 Hz** (deduplikasi, gap splitting)
2. Split continuous dataframe **sebelum** windowing
3. Sliding window dibuat **terpisah** dalam setiap split
4. Timestamp-key tensors disimpan dan diverifikasi overlap = 0

In [ ]:
# ============================================================
# Preprocessing v4: Split-Before-Windowing
# ============================================================
# File: src/preprocessing_v4.py

def to_strict_1hz_segments(df, source_id, profile_code_start, min_len=100):
    """
    1. Collapse duplicate seconds (keep first)
    2. Split at time gaps > 1s
    3. Each segment guaranteed strict 1 Hz
    4. Segments < min_len discarded
    """
    pass

def verify_no_split_leakage(splits):
    """
    Memverifikasi ZERO timestamp overlap antara splits.
    
    Checks:
      - train ∩ val  = ∅  (raw timestamps)
      - train ∩ test = ∅
      - val ∩ test   = ∅
      - Same checks on timestamp_keys (profile-specific)
    
    Jika ada overlap → assertion error, pipeline berhenti.
    """
    pass

# Pipeline flow:
# raw CSV → 1Hz segments → split_train_val(segment) → build_windows_v4(split)
# BUKAN: raw CSV → build_windows → split (LEAKAGE!)

### 3.4 Preprocessing v5: Contextual Anchor Features

**File**: `src/preprocessing_v5_contextual.py`

**Masalah yang diatasi**: Anchor head lemah pada −20°C OOD.

Pada suhu −20°C, model hanya melihat 100 timestep (100 detik) data dalam satu window. Tanpa konteks tambahan, anchor head kesulitan memprediksi SOC awal yang akurat karena:
- Tegangan terdistorsi oleh polarisasi tinggi
- Tidak ada informasi tentang **histori** perilaku baterai sebelumnya

**Solusi**: Menambahkan **14 fitur kontekstual kausal** (60 detik lookback) ke anchor head.

**14 Anchor Context Features:**

| Grup | Fitur | Deskripsi |
|------|-------|-----------|
| **OCV-Rest** (4) | `ctx_ocv_rest_voltage`, `ctx_ocv_rest_current`, `ctx_ocv_rest_soc`, `ctx_ocv_rest_valid` | Tegangan/arus/SOC saat rest terakhir |
| **History** (10) | `ctx_hist_V_proxy_mean_60s`, `..._std_60s`, `..._I_mean_60s`, `..._I_std_60s`, `..._T_mean_60s`, `..._dV_mean_60s`, `..._dI_mean_60s`, `..._delta_V_60s`, `..._delta_I_60s`, `ctx_hist_valid_60s` | Statistik 60s kausal dari V_proxy, I, T, derivatif |

In [ ]:
# ============================================================
# Preprocessing v5: Contextual Anchor Features
# ============================================================
# File: src/preprocessing_v5_contextual.py

ANCHOR_CTX_COLS = [
    # OCV-Rest group (4 features)
    'ctx_ocv_rest_voltage',      # Tegangan saat rest terakhir
    'ctx_ocv_rest_current',      # Arus saat rest (≈0)
    'ctx_ocv_rest_soc',          # SOC saat rest
    'ctx_ocv_rest_valid',        # Flag: apakah rest ditemukan?
    
    # History group (10 features)
    'ctx_hist_V_proxy_mean_60s', # Mean V_proxy 60 detik terakhir
    'ctx_hist_V_proxy_std_60s',  # Std V_proxy → variabilitas tegangan
    'ctx_hist_I_mean_60s',       # Mean arus → pola discharge/charge
    'ctx_hist_I_std_60s',        # Std arus → agresivitas driving
    'ctx_hist_T_mean_60s',       # Mean suhu → kondisi termal
    'ctx_hist_dV_mean_60s',      # Mean derivative tegangan
    'ctx_hist_dI_mean_60s',      # Mean derivative arus
    'ctx_hist_delta_V_60s',      # Delta V dalam 60s → trend
    'ctx_hist_delta_I_60s',      # Delta I dalam 60s → trend
    'ctx_hist_valid_60s',        # Flag: apakah 60s history tersedia?
]

# Indeks fitur OCV-Rest (yang di-zero-kan dalam mode history-only)
OCV_CTX_INDICES = [0, 1, 2, 3]
# Indeks fitur History (yang dipertahankan)
HISTORY_CTX_INDICES = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]

# KRITIS: Mode "history-only" → zero-out OCV-Rest features
# Alasan: Pada −20°C OOD, rest voltage tidak representatif
# karena polarisasi ekstrem. History features lebih robust.
def apply_history_only_masking(anchor_ctx):
    """Zero out OCV-rest features, retain history features."""
    masked = anchor_ctx.copy()
    masked[:, OCV_CTX_INDICES] = 0.0  # Kill OCV-rest stream
    return masked

---
<a id='4'></a>
## 4. Evolusi Arsitektur Model (v1 → v7)

Arsitektur model berevolusi melalui 7 versi:

```
v1 (model.py)              → TCN + Physics-Informed Loss (soft penalty)
  │   PVR ≠ 0%  ← masalah!
  ▼
v3 (model_v3.py)           → TCN + Hard Constraint (structural PVR=0%)
  │   MaxE = 99.90% pada −20°C ← cumulative drift collapse!
  ▼
v4 (model_v4_lstm.py)      → LSTM + Hard Constraint (ablation study)
  │   Sama: drift collapse
  ▼
v5 (model_v5_coulomb.py)   → Hard-COULOMB Constraint (bounded delta)
  │   MaxE turun drastis, PVR=0% dipertahankan
  ▼
v6 (model_v6_contextual.py)→ Contextual Hard-Coulomb LSTM
  │   −20°C t=0 RMSE membaik signifikan
  ▼
v7 (model_v7_gated_context.py) → Gated Contextual ← GAGAL!
  │   Learned gating degradasi performa
  ▼
v5_tcn (model_v5_coulomb_tcn.py) → Hard-Coulomb TCN (Sprint 52 redemption)
```

### 4.1 Model v1: TCN dengan Physics-Informed Loss

**File**: `src/model.py`

Arsitektur awal menggunakan TCN backbone dengan Physics-Informed Loss (soft penalty):

$$\mathcal{L} = \mathcal{L}_{MSE} + \lambda \cdot \mathcal{L}_{physics}$$

$$\mathcal{L}_{physics} = \frac{1}{N} \sum \text{ReLU}(\Delta SOC_t) \cdot \mathbb{1}[I_t < -0.05]$$

**Masalah**: Physics penalty hanya **mengurangi** pelanggaran, tidak **menghilangkan**. PVR > 0% pada test set.

In [ ]:
# ============================================================
# Model v1: TCN + Physics-Informed Loss
# ============================================================
# File: src/model.py

class TCN_SOC_Estimator(nn.Module):
    """
    Seq2Seq TCN: (B, 100, 5) → (B, 100, 1)
    
    Architecture:
      1. 4× TemporalBlock (causal conv + weight norm + LayerNorm)
      2. Linear head → Sigmoid → SOC ∈ [0, 1]
    
    Receptive field = 1 + 2×(K-1)×Σd = 1 + 2×6×15 = 181 steps
    """
    def forward(self, x):  # x: (B, 100, 5)
        h = self.tcn(x.transpose(1,2)).transpose(1,2)  # (B, 100, 64)
        return self.fc(h)  # (B, 100, 1) via Sigmoid

class PhysicsInformedLoss(nn.Module):
    """
    L = MSE + λ × Σ ReLU(ΔSOC) × 1[discharge]
    
    Problem: soft penalty → PVR > 0% at test time
    Solution: Hard Constraint (v3)
    """
    pass

### 4.2 Model v3: Hard Constraint (PVR = 0% Structural)

**File**: `src/model_v3.py`

**Inovasi kunci**: Mengganti soft penalty dengan **Hard Constraint layer** non-parametrik.

```
TCN Backbone → hidden states
    ├── Delta Head: Linear(64→32) → ReLU → Linear(32→1)  [tanpa Sigmoid]
    └── Anchor Head: Linear(64→16) → ReLU → Linear(16→1) → Sigmoid
        │
        ▼
Hard Constraint Layer:
    if I < -threshold (discharge): δSOC = -ReLU(-δSOC_raw) ≤ 0
    if I >  threshold (charge):    δSOC =  ReLU(δSOC_raw)  ≥ 0
    if |I| < threshold (rest):     δSOC = 0
    
    SOC(t) = SOC_anchor + Σ δSOC(1..t)    [clamped to [0,1]]
```

**Hasil**: PVR = 0.00% **dijamin secara struktural**, bukan learned.

**Masalah baru**: Cumulative drift collapse pada −20°C → MaxE = 99.90%

### 4.3 Model v5: Hard-Coulomb Constraint

**File**: `src/model_v5_coulomb.py`

**Inovasi kunci**: Menambahkan **magnitude bound** berdasarkan Hukum Coulomb pada setiap delta SOC.

Pada v3, delta SOC hanya dibatasi **arahnya** (sign), tapi tidak **besarnya**. Network bisa memprediksi perubahan SOC yang secara fisika mustahil (misal: 5% SOC drop dalam 1 detik pada arus 0.1A).

**Coulomb Bound**:

$$|\Delta SOC_{max}| = \frac{|I_t| \times \Delta t}{Q_{nom} \times 3600} \times \text{safety\_factor}$$

Pada 1 Hz (Δt = 1s) dengan Q_nom = 3.0 Ah:

$$|\Delta SOC_{max}| = \frac{|I_t|}{10800} \times 1.5$$

Safety factor 1.5 memberikan headroom 50% untuk noise dan transien.

In [ ]:
# ============================================================
# Model v5: HardCoulombConstraint
# ============================================================
# File: src/model_v5_coulomb.py

class HardCoulombConstraint(nn.Module):
    """
    Physics-bounded constraint layer.
    
    Combines:
      1. Direction clamp (v3): sign(δSOC) konsisten dengan arah arus
      2. Magnitude bound (NEW): |δSOC| ≤ |I|/(Q×3600) × safety_factor
    
    Ini mencegah cumulative drift yang menyebabkan MaxE = 99.90% pada v3.
    """
    def __init__(self, q_nominal=3.0, safety_factor=1.5):
        super().__init__()
        self.q_nominal = q_nominal
        self.safety_factor = safety_factor
    
    def forward(self, delta_raw, current_seq, soc_anchor):
        # Step 1: Compute Coulomb bound per timestep
        # delta_max = |I| / (Q_nom × 3600) × safety_factor
        
        # Step 2: Direction clamp (same as v3)
        # Discharge: δ ≤ 0, Charge: δ ≥ 0, Rest: δ = 0
        
        # Step 3: Magnitude clamp
        # |δ| = min(|δ_clamped|, δ_max)
        
        # Step 4: Cumulative sum
        # SOC(t) = anchor + Σ δ(1..t), clamped [0, 1]
        pass

class HardCoulombLSTM(nn.Module):
    """
    LSTM backbone + dual heads + HardCoulombConstraint.
    
    Forward:
      (B, 100, 5) + (B, 100) current → (B, 100, 1) SOC
    
    Trainable params: ~57k (vs TCN ~211k)
    """
    pass

### 4.4 Model v6: Contextual Hard-Coulomb LSTM

**File**: `src/model_v6_contextual.py`

**Inovasi kunci**: Anchor head menerima **static contextual features** selain hidden state LSTM.

```
LSTM Backbone → hidden: (B, T, 64)
    ├── Delta Head: Linear(64→32) → ReLU → Linear(32→1)
    │       ↓
    │   HardCoulombConstraint
    │
    └── Anchor Head:
            hidden[:, 0, :] (64-dim) ──┐
            anchor_ctx (14-dim) ──→ Encoder(14→32) → ReLU → LayerNorm ──┤
                                                                        ↓
                                                              Cat(64+32=96) → Linear(96→32) → ReLU → Linear(32→1) → Sigmoid
```

Anchor context: 14 fitur dari preprocessing v5 (4 OCV-rest + 10 history).

**Mode operasi**: `history_only` — OCV-rest features di-zero-kan karena tidak reliable pada −20°C.

In [ ]:
# ============================================================
# Model v6: ContextualHardCoulombLSTM
# ============================================================
# File: src/model_v6_contextual.py

class ContextualHardCoulombLSTM(nn.Module):
    """
    Hard-Coulomb LSTM dengan contextual anchor head.
    
    Inputs:
      x_seq      : (B, T, 5)   — sequence features
      current_seq: (B, T)      — unscaled current (Ampere)
      anchor_ctx : (B, 14)     — static contextual features
    
    Output:
      soc_pred   : (B, T, 1)   — SOC trajectory, PVR=0%
    """
    def __init__(self, num_inputs=5, anchor_ctx_dim=14,
                 hidden_size=64, num_layers=2, dropout=0.2,
                 safety_factor=1.5):
        super().__init__()
        self.lstm = nn.LSTM(num_inputs, hidden_size, num_layers,
                           batch_first=True, dropout=dropout)
        
        self.delta_head = nn.Sequential(
            nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
        
        # Contextual anchor encoder
        self.anchor_ctx_encoder = nn.Sequential(
            nn.Linear(anchor_ctx_dim, 32), nn.ReLU(), nn.LayerNorm(32))
        
        self.anchor_head = nn.Sequential(
            nn.Linear(hidden_size + 32, 32),  # 64 + 32 = 96
            nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
        
        self.hard_constraint = HardCoulombConstraint(
            q_nominal=3.0, safety_factor=safety_factor)
    
    def forward(self, x_seq, current_seq, anchor_ctx):
        hidden, _ = self.lstm(x_seq)  # (B, T, 64)
        delta_raw = self.delta_head(hidden)  # (B, T, 1)
        
        ctx = self.anchor_ctx_encoder(anchor_ctx)  # (B, 32)
        anchor_input = torch.cat([hidden[:, 0, :], ctx], dim=-1)
        soc_anchor = self.anchor_head(anchor_input)  # (B, 1)
        
        return self.hard_constraint(delta_raw, current_seq, soc_anchor)

### 4.5 Model v7: Gated Context — GAGAL ❌

**File**: `src/model_v7_gated_context.py`

**Hipotesis**: Learned gating antara OCV-rest dan history streams akan lebih baik dari simple concatenation.

```
anchor_ctx → split → OCV-rest stream → encoder
                   → History stream   → encoder
                                          ↓
                   g = sigmoid(W[ocv_emb, hist_emb]) × ctx_ocv_rest_valid
                   ctx = g × ocv_emb + (1-g) × hist_emb
```

**Hasil**: Performa DEGRADASI signifikan dibandingkan v6.

**Kesimpulan**: Learned fusion lebih **tidak robust** dibandingkan simple concatenation + zero-masking. Gating mechanism menambah kompleksitas tanpa manfaat. Fitur history sudah cukup informatif tanpa perlu weighted combination.

**Status**: DITINGGALKAN. Dipertahankan sebagai bukti ablation study.

### 4.6 Hard-Coulomb TCN (Sprint 52 Redemption)

**File**: `src/model_v5_coulomb_tcn.py` + `src/sprint52_tcn_redemption.py`

Setelah LSTM terbukti lebih baik di v5/v6, Sprint 52 memberikan TCN backbone kesempatan kedua menggunakan:
- **Canonical HardCoulombConstraint** dari `model_v5_coulomb.py`
- **Contextual anchor head** (same architecture as v6 LSTM)
- History-only masking protocol

**Hasil**: TCN memberikan worst-case OOD performance terbaik (MaxE lebih rendah), tapi parameter count ~4× LSTM (~211k vs ~57k).

---
<a id='5'></a>
## 5. Ekstraksi R_int dari HPPC

**File**: `src/hppc_rint_extractor.py`

Script ini merupakan **proof of reproducibility** yang bisa diverifikasi reviewer.

### Metodologi

Resistansi Ohmic ($R_0$) diekstrak dari transisi REST → Discharge pada profil HPPC:

$$R_{int} = \frac{\Delta V}{\Delta I} = \frac{V_{pulse\_first} - V_{rest\_last}}{I_{pulse\_first} - I_{rest\_last}}$$

**Kriteria deteksi transisi:**
- Rest: $|I| \leq 0.05$ A
- Discharge pulse onset: $I \leq -0.10$ A
- Valid R_int: $0 < R_{int} \leq 0.50$ Ω

**Mengapa sample pertama?** Menggunakan sample yang delayed 1s akan meng-include dinamika polarisasi/difusi dan tidak mereproduksi nilai paper.

In [ ]:
# ============================================================
# HPPC R_int Extractor — Reproducibility Proof
# ============================================================
# File: src/hppc_rint_extractor.py

# Hasil Verifikasi (semua PASS, Δ < 0.01 mΩ):
#
# Temp     | Script (mΩ) | Paper (mΩ) | Δ (mΩ)  | Status
# ---------|-------------|------------|---------|-------
# 40°C     | 16.51       | 16.51      | +0.00   | PASS
# 25°C     | 19.86       | 19.86      | +0.00   | PASS
# 10°C     | 28.75       | 28.75      | +0.00   | PASS
# 0°C      | 40.08       | 40.08      | +0.00   | PASS
# -10°C    | 62.19       | 62.19      | +0.00   | PASS
# -20°C    | 109.83      | 109.83     | +0.00   | PASS
#
# Pulses per temperature: 45-60 REST→DCH transitions

---
<a id='6'></a>
## 6. Pipeline Training Final

Training framework final dibangun dalam Sprint 48–52 dengan arsitektur modular.

### 6.1 Sprint 48: Common Training Framework

**File**: `src/sprint48_common.py`

Framework training yang dipakai bersama oleh semua sprint final:
- AdamW optimizer
- CosineAnnealingLR scheduler
- Early stopping (patience=12) pada pure MSE validation loss
- Gradient clipping (max_norm=1.0)
- Reproducibility seeding
- Loss: **Pure MSE** (Hard Constraint menangani physics secara struktural)

In [ ]:
# ============================================================
# Sprint 48: Common Training Framework
# ============================================================
# File: src/sprint48_common.py

# Training configuration:
TRAINING_CONFIG = {
    'optimizer': 'AdamW',
    'lr': 1e-3,
    'scheduler': 'CosineAnnealingLR',
    'eta_min': 1e-6,
    'loss': 'MSELoss',           # Pure MSE — no physics penalty needed!
    'early_stopping': True,
    'patience': 12,
    'grad_clip': 1.0,
    'safety_factor': 1.5,        # Coulomb bound headroom
    'seed': 42,
}

# Entry points:
# sprint48_train_scenario_A.py → HardCoulombLSTM on Scenario A
# sprint48_train_scenario_B.py → HardCoulombLSTM on Scenario B

### 6.2 Sprint 50: Contextual LSTM Training

**File**: `src/sprint50_train_contextual.py`

Training ContextualHardCoulombLSTM dengan:
- Data v5_contextual (dengan anchor context features)
- History-only masking: OCV-rest features di-zero-kan
- Evaluasi fokus pada −20°C OOD performance

### 6.3 Sprint 52: TCN Redemption

**File**: `src/sprint52_tcn_redemption.py`

TCN backbone diberi kesempatan kedua dengan:
- Milestone 1: Clean v4 baseline (HardCoulombTCN pada v4 data)
- Milestone 2: Contextual history-only (ContextualHardCoulombTCN pada v5 data)
- Menggunakan canonical HardCoulombConstraint dari model_v5_coulomb.py

---
<a id='7'></a>
## 7. Evaluasi & Metrik

### 7.1 Metrik Utama

| Metrik | Formula | Deskripsi |
|--------|---------|----------|
| **RMSE (Full-Seq)** | $\sqrt{\frac{1}{NT}\sum(\hat{y}-y)^2}$ | Error rata-rata seluruh 100 timestep |
| **RMSE (Last-Step)** | $\sqrt{\frac{1}{N}\sum(\hat{y}_{t=-1}-y_{t=-1})^2}$ | Error pada timestep terakhir (backward-compatible) |
| **MAE** | $\frac{1}{NT}\sum|\hat{y}-y|$ | Mean Absolute Error |
| **MaxE** | $\max|\hat{y}-y|$ | Worst-case error (kritis untuk safety) |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Coefficient of determination |
| **PVR** | $\frac{\text{violations}}{\text{discharge steps}} \times 100\%$ | Physics Violation Rate |

### 7.2 Physics Violation Rate (PVR)

PVR mengukur seberapa sering model melanggar hukum fisika:

In [ ]:
# ============================================================
# PVR Computation (Intra-Window, Seq2Seq)
# ============================================================

def compute_pvr(y_pred, current_values):
    """
    Physics Violation Rate:
    
    Violation = ΔSOC > 0 KETIKA baterai sedang discharge (I < -0.05A)
    
    Secara fisika, SOC HARUS turun saat discharge.
    Jika model memprediksi SOC naik → physics violation.
    
    Hard-Coulomb Constraint MENJAMIN PVR = 0.00%
    karena δSOC dikunci ≤ 0 saat discharge.
    """
    # Intra-window: delta across timesteps within each window
    delta_soc = y_pred[:, 1:] - y_pred[:, :-1]    # (N, T-1)
    discharge_mask = current_values[:, 1:] < -0.05  # (N, T-1)
    
    violations = (delta_soc > 0) & discharge_mask
    n_violations = violations.sum()
    n_discharge = discharge_mask.sum()
    
    pvr = (n_violations / n_discharge) * 100.0
    return pvr  # Untuk Hard-Coulomb: selalu 0.00%

### 7.3 Per-Temperature Breakdown

Untuk Scenario A, evaluasi dilakukan per suhu test:

| Suhu | Status | Tantangan |
|------|--------|----------|
| 40°C | OOD (mild) | Sedikit di luar range training |
| −10°C | OOD (moderate) | R_int 3× lebih besar dari training |
| **−20°C** | **OOD (extreme)** | **R_int 6.6× lebih besar, observability bottleneck** |

---
<a id='8'></a>
## 8. Penemuan Kritis: Anchor Trap −20°C

### 8.1 Fenomena

Pada −20°C (suhu paling ekstrem, sepenuhnya OOD), model tanpa contextual features menunjukkan **anomali anchor**:

- Anchor head memprediksi SOC awal yang **sangat tidak akurat**
- Karena HardCoulombConstraint membatasi perubahan SOC per timestep, trajectory SOC "terjebak" di sekitar anchor yang salah
- Hasil: t=0 RMSE dan MaxE sangat tinggi pada −20°C

### 8.2 Root Cause: Empirical Observability Bottleneck

Pada −20°C:
1. R_int = 109.83 mΩ (6.6× dari 40°C)
2. Tegangan terminal sangat terdistorsi oleh polarisasi
3. Meskipun V_proxy mengoreksi Ohmic drop, **polarisasi dinamis** tetap tinggi
4. Model hanya melihat 100 timestep — **tidak cukup konteks** untuk anchor head

### 8.3 Solusi: Causal History Injection

Menambahkan 60 detik **causal history features** ke anchor head:

```
TANPA contextual features:
  Anchor input = hidden[:, 0, :]  (64-dim)
  → Hanya melihat representasi LSTM dari 100 timestep saat ini
  → TIDAK CUKUP pada −20°C

DENGAN contextual features:
  Anchor input = [hidden[:, 0, :], ctx_encoder(anchor_ctx)]  (64+32 = 96-dim)
  → Melihat representasi LSTM + statistik 60 detik SEBELUM window
  → Mean/std V_proxy, I, T dari 60s terakhir
  → CUKUP untuk mengestimasi SOC awal bahkan pada −20°C
```

### 8.4 Kunci: History-Only Mode

Eksperimen menunjukkan bahwa **hanya** history features yang efektif:
- OCV-rest features: **TIDAK reliable** pada −20°C karena rest voltage belum equilibrium
- History features: **ROBUST** karena berbasis statistik temporal (trend, variabilitas)

Maka mode operasi final: **history_only** — OCV-rest features di-zero-kan.

---
<a id='9'></a>
## 9. Hasil Akhir & Rekomendasi Deployment

### 9.1 Perbandingan Arsitektur Final

| Model | Params | PVR | Kelebihan | Kelemahan |
|-------|--------|-----|-----------|----------|
| **Contextual HC-LSTM** | ~57k | 0.00% | Ringan, cocok edge device | MaxE sedikit lebih tinggi |
| **Contextual HC-TCN** | ~211k | 0.00% | Worst-case OOD terbaik | 4× lebih besar |
| Gated Context (v7) | ~65k | 0.00% | — | **GAGAL**: degradasi performa |

### 9.2 Rekomendasi Deployment

| Skenario Deployment | Model Rekomendasi | Alasan |
|---------------------|-------------------|--------|
| **Edge BMS** (mikrokontroler, real-time) | Contextual HC-LSTM (~57k) | Parameter kecil, inference cepat |
| **Cloud/Server** (maximum safety) | Contextual HC-TCN (~211k) | Worst-case MaxE terendah |
| **Ensemble** (redundancy) | Keduanya | Cross-check prediction untuk safety-critical |

### 9.3 Kontribusi Utama

1. **HardCoulombConstraint**: Layer non-parametrik yang **menjamin PVR = 0%** secara struktural
2. **V_proxy**: Koreksi Ohmic drop menggunakan R_int yang di-pre-compute dari HPPC
3. **Contextual Anchor**: 60s causal history features mengatasi Anchor Trap pada suhu OOD
4. **History-Only Mode**: OCV-rest features di-zero-kan karena tidak reliable pada −20°C
5. **Split-Before-Windowing**: Eliminasi temporal leakage dalam pipeline preprocessing

### 9.4 Dashboard Demonstrasi

Dua dashboard Streamlit disediakan untuk demonstrasi:

1. **`src/web_app.py`** — Dashboard dasar untuk inference cepat
2. **`src/bms_dashboard.py`** — Dashboard premium dengan:
   - Dark automotive theme
   - Interactive Plotly visualizations
   - Safety audit panel dengan KPI metrics
   - Support upload CSV custom atau profil pre-loaded

```bash
# Menjalankan dashboard:
streamlit run src/bms_dashboard.py
```

---
<a id='10'></a>
## 10. Kesimpulan

### 10.1 Ringkasan

Penelitian ini mengembangkan arsitektur **Contextual Hard-Coulomb LSTM/TCN** untuk estimasi SOC baterai Li-Ion dengan jaminan keamanan fisika:

1. **PVR = 0.00%** dijamin secara struktural oleh HardCoulombConstraint
2. **Generalisasi suhu ekstrem** dicapai melalui V_proxy dan contextual history features
3. **Eliminasi temporal leakage** melalui split-before-windowing pipeline
4. **Reprodusibilitas penuh**: R_int values, preprocessing pipeline, dan training framework semuanya deterministic

### 10.2 Pelajaran Penting

| No | Pelajaran | Detail |
|----|-----------|--------|
| 1 | Soft penalty < Hard constraint | Physics-Informed Loss (v1) tidak menjamin PVR=0% |
| 2 | Direction clamp tidak cukup | Perlu magnitude bound (Coulomb) untuk mencegah drift |
| 3 | Simple > Complex | Concatenation > Learned gating (v6 > v7) |
| 4 | Causal history matters | 60s lookback mengatasi observability bottleneck |
| 5 | Split before windowing | Sliding window overlap = temporal leakage |

### 10.3 Future Work

- Ekspansi ke cell chemistry lain (LFP, NCA)
- Online learning untuk adaptasi SOH degradasi
- Quantization untuk deployment pada mikrokontroler BMS
- Validasi pada data real-world (bukan lab controlled)

---

### Struktur File Final

```
src/
├── config.py                          # Konfigurasi & konstanta
├── preprocessing_v4.py                # Leakage-safe 1Hz pipeline
├── preprocessing_v5_contextual.py     # Contextual anchor features
├── model_v5_coulomb.py                # ★ HardCoulombConstraint (core)
├── model_v5_coulomb_tcn.py            # Hard-Coulomb TCN backbone
├── model_v6_contextual.py             # ★ Contextual HC-LSTM (final)
├── model_v7_gated_context.py          # Gated context (ablation: gagal)
├── hppc_rint_extractor.py             # R_int reproducibility proof
├── sprint48_common.py                 # Shared training framework
├── sprint48_train_scenario_A.py       # Scenario A entry point
├── sprint48_train_scenario_B.py       # Scenario B entry point
├── sprint48_evaluate_all.py           # Comprehensive evaluation
├── sprint48_safety_ablation.py        # Safety factor ablation
├── sprint50_train_contextual.py       # ★ Contextual LSTM training
├── sprint51_train_gated.py            # Gated training (ablation)
├── sprint52_tcn_redemption.py         # ★ TCN final training
├── web_app.py                         # Streamlit dashboard (basic)
└── bms_dashboard.py                   # Streamlit dashboard (premium)
```

---

*Notebook ini dibuat sebagai dokumentasi lengkap penelitian. Tidak perlu dijalankan.*